In [1]:
%load_ext cudf.pandas
%load_ext cuml.accel

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

DEVICE = 'cuda' if torch.cuda.is_available else 'cpu'
DEVICE

'cuda'

In [ ]:
df = pd.read_csv('datasets/Quora Questions Pair Dataset/09_Xy.csv')
X = df.iloc[:,:-1]
y = df.iloc[:,-1]

In [5]:
class CustomDataset(Dataset):
    def __init__(self, q1, q2, y):
        self.q1 = torch.tensor(q1, dtype=torch.long)
        self.q2 = torch.tensor(q2, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.float)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.q1[idx], self.q2[idx], self.y[idx]

In [6]:
class model_gru(nn.Module):
    def __init__(self, embed_weight, hidden_dim=128):
        super().__init__()
        vocab_size, embed_dim = embed_weight.shape
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.embed.weight.data.copy_(embed_weight)
        self.embed.weight.requires_grad = True

        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Sequential(
            nn.Linear(hidden*4+2, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128,1)
        )
    def encode(self, X):
        embeb = self.embed(X)
        _, h = self.gru(embed)
        h = torch.cat([h[0], h[1]], dim=1)
        return h
    def forward(self, q1, q2):
        h1, h2 = self.encode(q1), self.encode(q2)
        cos = F.cosine_similarity(h1, h2).unsqueeze(1)
        l1 = torch.abs(h1 - h2).sum(1, keepdim=True)
        return self.fc(combined).squeeze(1)

In [ ]:
train_ds = CustomDataset()